Copyright 2026, Battelle Energy Alliance, LLC, ALL RIGHTS RESERVED

In [1]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os
os.chdir("../../..")
from HelpingFunctions import ERCOTProcessor
from HelpingFunctions import WeatherProcessing
from HelpingFunctions import FeatureEngineering
from HelpingFunctions import ForecastingHelpers

import onnxruntime as ort
ort.set_default_logger_severity(4)

In [2]:
import os 
os.getcwd()

'/home/ortild/Amaranth/opensourcegridmodeling'

Load Data

In [3]:
full_df = pd.read_csv('ElectricityDemandAustinTX/LoadForecastingAttacks/full_data.csv', parse_dates=['time'], index_col=['time']) 
hourly_res_norm = ForecastingHelpers.hourlyresiduals(full_df) 

/home/ortild/Amaranth/opensourcegridmodeling/ElectricityDemandAustinTX/LoadForecastingAttacks/HelpingFunctions/ForecastingHelpers.py:74: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  hourly_res_norm['load'] = df_norm['load'].groupby(pd.Grouper(freq='M')).transform(lambda x: x - x.mean())


GBR Model

In [4]:
import onnx

def get_tree_attributes(onnx_file):
    onnx_model =  onnx.load(onnx_file)
    tree_ensemble_node = None
    for node in onnx_model.graph.node:
        if node.op_type == "TreeEnsembleRegressor":
            tree_ensemble_node = node
            break
    
    if tree_ensemble_node is None:
        raise ValueError("No TreeEnsembleRegressor found in the ONNX graph.")
        
    attributes = {}
    # Inspect the model graph's attributes
    for attr in tree_ensemble_node.attribute:
        if attr.name == "nodes_modes":
            attributes[attr.name] = [s.decode('utf-8') for s in attr.strings]
        elif attr.type == onnx.AttributeProto.AttributeType.FLOATS:
            attributes[attr.name] = list(attr.floats)
        elif attr.type == onnx.AttributeProto.AttributeType.INTS:
            attributes[attr.name] = list(attr.ints)
        elif attr.name == "post_transform":
            attributes[attr.name] = attr.s.decode('utf-8')
    return attributes


In [5]:
class TreeNode:
    def __init__(self, node_id, mode, feature_id=None, threshold=None, true_child=None, false_child=None, leaf_value=None):
        self.node_id = node_id
        self.mode = mode
        self.feature_id = feature_id
        self.threshold = threshold
        self.true_child = true_child
        self.false_child = false_child
        self.leaf_value = leaf_value
        
class DecisionTree:
    def __init__(self, tree_id, node_map):
        self.tree_id = tree_id
        self.node_map = node_map
        self.root = self.node_map[0]
    
        
    def predict(self, x):
        """Walk the tree to find the prediction for a single sample."""
        node = self.root
        while node.mode != 'LEAF':
            feature_value = x[node.feature_id]
            
            # Note: handle different node modes (LEQ, LT, etc.)
            if node.mode == 'BRANCH_LEQ':
                if feature_value <= node.threshold:
                    node = node.true_child
                else:
                    node = node.false_child
            # Add other modes as needed
            else:
                raise NotImplementedError(f"Unsupported node mode: {node.mode}")
        
        return node.leaf_value

In [6]:
def reconstruct_gbr_model(attributes):
    """
    Reconstructs the full GBR model from the parsed ONNX attributes.
    """
    all_trees = []
    
    # Group nodes by tree
    nodes_by_tree = {}
    for i, tree_id in enumerate(attributes['nodes_treeids']):
        if tree_id not in nodes_by_tree:
            nodes_by_tree[tree_id] = []
        nodes_by_tree[tree_id].append(i)

        
    # Create a mapping for leaf weights
    leaf_weight_map = {}
    for i, (tree_id, node_id) in enumerate(zip(attributes['target_treeids'], attributes['target_nodeids'])):
        if (tree_id, node_id) not in leaf_weight_map:
            leaf_weight_map[(tree_id, node_id)] = attributes['target_weights'][i]

    # Reconstruct each individual tree
    for tree_id, indices in nodes_by_tree.items():
        node_map = {}
        # Create TreeNode objects
        for i in indices:
            node_id = attributes['nodes_nodeids'][i]
            mode = attributes['nodes_modes'][i]
            
            if mode == 'LEAF':
                leaf_value = leaf_weight_map.get((tree_id, node_id))
                node_map[node_id] = TreeNode(node_id, mode, leaf_value=leaf_value)
            else: # It's a branch node
                feature_id = attributes['nodes_featureids'][i]
                threshold = attributes['nodes_values'][i]
                node_map[node_id] = TreeNode(node_id, mode, feature_id=feature_id, threshold=threshold)
        
        # Link children
        for i in indices:
            node_id = attributes['nodes_nodeids'][i]
            mode = attributes['nodes_modes'][i]
            if mode != 'LEAF':
                true_child_id = attributes['nodes_truenodeids'][i]
                false_child_id = attributes['nodes_falsenodeids'][i]
                node_map[node_id].true_child = node_map[true_child_id]
                node_map[node_id].false_child = node_map[false_child_id]
        
        all_trees.append(DecisionTree(tree_id, node_map))

    return all_trees, attributes['base_values']


In [7]:
# Get the attributes of the GBR model from the onnx file
onnx_file = "ElectricityDemandAustinTX/LoadForecastingAttacks/SaveFiles/gbr_no_mapie.onnx"
attributes = get_tree_attributes(onnx_file)

# Reconstruct the trees and get base values
trees, base_values = reconstruct_gbr_model(attributes)

# Use the reconstructed model for prediction
def predict_gbr(x, trees, base_values):
    prediction = base_values[0]
    for tree in trees:
        prediction += tree.predict(x)
    return prediction

In [9]:
import random
index = 0
synthData = []
while index <= 100000:
    random_vector =  []
    for i in range(0,13):
        random_number = random.uniform(0, 1)
        random_vector.append(random_number)
    synthData.append(random_vector)
    index+=1
#print(synthData)


In [10]:
# SET UP DATA INPUT TO TRY TO FIND REAL INPUT DATA BASED ON OUTPUTS
testX = synthData
input_df = pd.DataFrame(testX, columns=["inputA","inputB", "inputC", "inputD", "inputE", "inputF", "inputG", "inputH", "inputI", "inputJ", "inputK", "inputL", "inputM"])
print(input_df)
knownOutput = -0.14729762 

          inputA    inputB    inputC    inputD    inputE    inputF    inputG  \
0       0.587111  0.965258  0.968382  0.729121  0.704509  0.096440  0.188213   
1       0.384156  0.809170  0.879363  0.312022  0.771119  0.635947  0.505988   
2       0.425925  0.085226  0.999078  0.521709  0.084887  0.994836  0.325542   
3       0.373131  0.903090  0.478667  0.519435  0.543934  0.636472  0.924226   
4       0.649961  0.204519  0.486077  0.770228  0.719102  0.460813  0.350338   
...          ...       ...       ...       ...       ...       ...       ...   
99996   0.668952  0.887548  0.984626  0.737401  0.642055  0.074469  0.251491   
99997   0.432330  0.347922  0.648285  0.960648  0.570068  0.161484  0.959516   
99998   0.877755  0.299493  0.108137  0.644301  0.941986  0.269212  0.946470   
99999   0.020985  0.050424  0.278784  0.154965  0.856323  0.095173  0.726080   
100000  0.397732  0.173146  0.701285  0.816404  0.139454  0.834802  0.102813   

          inputH    inputI    inputJ   

In [11]:
## test prediction 
final_prediction = []
for inputs in testX:
    output = predict_gbr(inputs, trees, base_values)
    final_prediction.append(output)
output_df = pd.DataFrame(final_prediction, columns=["output"])
data_df = pd.concat([input_df, output_df], axis = 1)
data_df.head(5)

,inputA,inputB,inputC,inputD,inputE,inputF,inputG,inputH,inputI,inputJ,inputK,inputL,inputM,output
0,0.587111,0.965258,0.968382,0.729121,0.704509,0.096440,0.188213,0.838026,0.116683,0.848037,0.154696,0.107198,0.667170,0.302084
1,0.384156,0.809170,0.879363,0.312022,0.771119,0.635947,0.505988,0.282777,0.171422,0.409662,0.224737,0.337305,0.274744,0.313065
2,0.425925,0.085226,0.999078,0.521709,0.084887,0.994836,0.325542,0.824028,0.848503,0.342688,0.145186,0.067485,0.258965,0.327476
3,0.373131,0.903090,0.478667,0.519435,0.543934,0.636472,0.924226,0.031148,0.074478,0.511542,0.214983,0.965699,0.506089,0.351045
4,0.649961,0.204519,0.486077,0.770228,0.719102,0.460813,0.350338,0.896342,0.479046,0.511505,0.603115,0.973558,0.306302,0.388790


Create a GAN to inverse the GBR model, taking in the outputs and predicting a vector of the 13 inputs - based on "A GAN based solver of black-box inverse problems"


In [17]:
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error


x = data_df["output"].to_numpy().reshape(-1,1)
y = data_df.drop(columns="output").to_numpy()
X_train, X_test, y_train, y_test = train_test_split(x, y, random_state=42, train_size = .8)

invModel = MLPRegressor(hidden_layer_sizes=(500, 30), max_iter=5000)
# Train the model
invModel.fit(X_train, y_train)

y_pred = invModel.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f"Mean Absolute Error (MAE): {mae}")


Mean Absolute Error (MAE): 0.2357881663047543


In [18]:
knownOutput = [[-0.14729762]] #for testing?
predicted_vector = invModel.predict(knownOutput)
length = len(predicted_vector[0])
for index in range(length):
    if index > 2:
        predicted_vector[0][index] = round(predicted_vector[0][index])
    index +=1
print(predicted_vector)

[[-0.08243353  0.61391428  0.55635485  1.          1.          0.
   0.          1.          1.          1.          1.          0.
   1.        ]]


In [14]:
# train-validate-test split
train = hourly_res_norm[:'2014']
validate = hourly_res_norm['2015':'2016']
test = hourly_res_norm['2017':]

# setup training variables 
exog_tr = train.iloc[:,1:].values
ar_tr = train['load'].shift().bfill().values[:,None]
X_tr = np.hstack([ar_tr, exog_tr])
y_tr = train['load'].values

# setup validation variables
exog_val = validate.iloc[:,1:].values
y_val = validate['load'].values

# setup testing variables
exog_te = test.iloc[:,1:].values
ar_test = test['load'].shift().bfill().values[:,None]
y_test = test['load'].values
X_test = np.hstack([ar_test, exog_te])

# setup miscellaneous variables
yp_full = hourly_res_norm.loc[:'2016','load']
yp_val = hourly_res_norm.loc['2015':'2016','load']
yp_te = hourly_res_norm.loc['2017':,'load']
y_init_val = np.hstack([y_tr[-1], validate.iloc[167::168,0].values])
y_init_te = np.hstack([y_val[-1], test.iloc[167::168,0].values])

In [15]:
# Create test value
manualX = X_test[0]
manualY = y_test[0]


[-0.09924674  0.48484848  0.93        0.          0.          0.
  1.          0.          0.          0.          0.          0.
  1.        ]


In [16]:
# KNOWN OUTPUT FOR THIS CASE THAT WE ARE TRYING TO FIND THE INPUTS FOR
inputsL = []
inputsL.append(manualX.tolist())
#print(inputsL)

import onnxruntime as rt
onnx_model =  onnx.load("ElectricityDemandAustinTX/LoadForecastingAttacks/SaveFiles/gbr_no_mapie.onnx")
model_inv = onnx_model
session = rt.InferenceSession("ElectricityDemandAustinTX/LoadForecastingAttacks/SaveFiles/gbr_no_mapie.onnx")
input_name = session.get_inputs()[0].name
input_data = np.array(inputsL, dtype=np.float32) 
#input_data = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
inputs = {input_name: input_data}
outputs = session.run(None, inputs)
print(outputs)

[array([[-0.14729762]], dtype=float32)]
